# Notebook 08: Figures and Tables

**Purpose**: Generate all thesis figures and summary tables from the results parquets.

## Why this notebook is last, and reads-only

Every figure here is a rendering of a number already computed and saved by an earlier notebook — this notebook performs no new inference, no new statistics beyond significance testing, and no new metric definitions. It exists purely to translate `results/*.parquet` into the specific artefacts (`figures/*.pdf`, `tables/*.csv`) a thesis document consumes, one per research question. If a figure here looks wrong, the fix is almost always upstream (re-run the notebook that produced the underlying parquet), not in this notebook's plotting code.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.results_schema import load_results

sns.set_theme(style='whitegrid')

resolve_path('figures').mkdir(parents=True, exist_ok=True)
resolve_path('tables').mkdir(parents=True, exist_ok=True)


def safe_read_parquet(path):
    """None if the file doesn't exist *or* exists but has no rows (e.g. an
    upstream notebook ran with vLLM unavailable and only wrote the empty
    schema) -- callers can use a single `is None` check either way."""
    try:
        df = pd.read_parquet(resolve_path(path))
    except FileNotFoundError:
        return None
    return df if not df.empty else None

## Figures

1. RQ1 bar chart: ID vs OOD accuracy, 3 datasets, zero-shot + random-k
2. RQ2 heatmap: protocol x shift type accuracy grid (synthetic arm)
3. RQ3 scatter: Delta-accuracy vs Delta-rho across conditions
4. RQ4 grouped bars: SATA+protocol vs protocol vs random, accuracy + rho, bootstrap CI error bars
5. Retention curves: R-AUC plots, best vs worst conditions
6. SATA ablation table: full vs query-agnostic vs best-protocol-alone
7. Training curves: SATA loss + validation metric over epochs
8. k-sensitivity table/figure: accuracy at k_primary vs k_sensitivity, per method, both arms

Each figure maps to exactly one research question's success criterion (Lit-review §3), so reading these off in order tells the project's whole story: (1) does the OOD problem exist at all — Gate 1; (2) which demonstration diversity helps, and does it interact with shift type — RQ2; (3) does accuracy ever improve without faithfulness improving alongside it — RQ3's central dissociation question; (4) does SATA's learned, query-conditioned reweighting beat the best hand-designed protocol on *both* axes — RQ4; (5)/(7) supporting evidence for calibration and training dynamics; (6) isolates which part of SATA's design (query-conditioning, in particular) is responsible for any gain seen in (4); (8) checks whether every headline number above (all computed at k_primary) is a stable ranking or an artefact of one specific demo count.

In [ ]:
rq1_summary = safe_read_parquet('results/real_arm_baselines_summary.parquet')
if rq1_summary is None:
    print("Skipping RQ1 figure — results/real_arm_baselines_summary.parquet not found (run Notebook 02 first).")
else:
    # Notebook 02 now sweeps k_primary + k_sensitivity, so this file has a 'k'
    # column with 2 rows per (dataset, model, method, environment) -- without
    # filtering, seaborn would silently average both k values into one bar.
    if 'k' in rq1_summary.columns:
        rq1_summary = rq1_summary[rq1_summary['k'] == config.k_primary]
    rq1_data = rq1_summary[rq1_summary['method'].isin(['zero_shot', 'random'])].copy()
    rq1_data['condition'] = rq1_data['method'] + ' / ' + rq1_data['environment']

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=rq1_data, x='dataset', y='accuracy_mean', hue='condition', ax=ax)
    ax.set_ylabel('Accuracy')
    ax.set_title('RQ1: ID vs OOD accuracy (zero-shot, random-k)')
    fig.tight_layout()
    fig.savefig(resolve_path('figures/rq1_ood_degradation.pdf'))
    plt.close(fig)
    print("Saved figures/rq1_ood_degradation.pdf")

In [4]:
rq2_grid = safe_read_parquet('results/rq2_grid.parquet')
if rq2_grid is None:
    print("Skipping RQ2 figure — results/rq2_grid.parquet not found (run Notebook 06 first).")
else:
    pivot = rq2_grid.pivot_table(index='method', columns='shift_type', values='accuracy')
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis', ax=ax)
    ax.set_title('RQ2: protocol x shift type accuracy (synthetic arm)')
    fig.tight_layout()
    fig.savefig(resolve_path('figures/rq2_protocol_shift_heatmap.pdf'))
    plt.close(fig)
    print("Saved figures/rq2_protocol_shift_heatmap.pdf")

Skipping RQ2 figure — results/rq2_grid.parquet not found (run Notebook 06 first).


In [ ]:
real_acc = safe_read_parquet('results/real_arm_baselines_summary.parquet')
real_rho = safe_read_parquet('results/faithfulness_real_rho_summary.parquet')
syn_rho_raw = safe_read_parquet('results/faithfulness_synthetic.parquet')
rq2_for_rq3 = safe_read_parquet('results/rq2_grid.parquet')

# real_acc now has a 'k' column (Notebook 02 sweeps k_primary + k_sensitivity)
# -- filter to k_primary so the .iloc[0]/groupby lookups below don't grab an
# arbitrary k row or double-plot each method once per k value. rq2_for_rq3
# needs no such filter: Notebook 06 already writes rq2_grid.parquet
# k_primary-only.
if real_acc is not None and 'k' in real_acc.columns:
    real_acc = real_acc[real_acc['k'] == config.k_primary]

scatter_rows = []

if real_acc is not None and real_rho is not None:
    real_acc_ood = real_acc[real_acc['environment'] == 'ood']
    for (dataset, model), grp in real_acc_ood.groupby(['dataset', 'model']):
        base_acc_row = grp[grp['method'] == 'random']['accuracy_mean']
        rho_grp = real_rho[(real_rho['dataset'] == dataset) & (real_rho['model'] == model)]
        base_rho_row = rho_grp[rho_grp['method'] == 'random']['rho_mean']
        if base_acc_row.empty or base_rho_row.empty:
            continue
        base_acc, base_rho = base_acc_row.iloc[0], base_rho_row.iloc[0]
        for _, row in grp.iterrows():
            rho_row = rho_grp[rho_grp['method'] == row['method']]['rho_mean']
            if rho_row.empty:
                continue
            scatter_rows.append({
                'arm': 'real', 'method': row['method'],
                'delta_accuracy': row['accuracy_mean'] - base_acc,
                'delta_rho': rho_row.iloc[0] - base_rho,
            })

if rq2_for_rq3 is not None and syn_rho_raw is not None:
    syn_rho_summary = syn_rho_raw.groupby(['model', 'method'])['rho'].mean().reset_index()
    for model, grp in rq2_for_rq3.groupby('model'):
        id_grp = grp[grp['shift_type'] == 'id']
        base_acc_row = id_grp[id_grp['method'] == 'random']['accuracy']
        base_rho_row = syn_rho_summary[(syn_rho_summary['model'] == model) & (syn_rho_summary['method'] == 'random')]['rho']
        if base_acc_row.empty or base_rho_row.empty:
            continue
        base_acc, base_rho = base_acc_row.iloc[0], base_rho_row.iloc[0]
        for _, row in id_grp.iterrows():
            rho_row = syn_rho_summary[(syn_rho_summary['model'] == model) & (syn_rho_summary['method'] == row['method'])]['rho']
            if rho_row.empty:
                continue
            scatter_rows.append({
                'arm': 'synthetic', 'method': row['method'],
                'delta_accuracy': row['accuracy'] - base_acc,
                'delta_rho': rho_row.iloc[0] - base_rho,
            })

if not scatter_rows:
    print("Skipping RQ3 figure — need both accuracy and faithfulness results (Notebooks 02/03 and/or 06).")
else:
    scatter_df = pd.DataFrame(scatter_rows)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(data=scatter_df, x='delta_accuracy', y='delta_rho', hue='arm', style='method', s=100, ax=ax)
    ax.axhline(0, color='grey', linewidth=0.8)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('Delta accuracy (vs random)')
    ax.set_ylabel('Delta rho (vs random)')
    ax.set_title('RQ3: accuracy vs faithfulness dissociation')
    fig.tight_layout()
    fig.savefig(resolve_path('figures/rq3_accuracy_vs_faithfulness.pdf'))
    plt.close(fig)
    print("Saved figures/rq3_accuracy_vs_faithfulness.pdf")

In [6]:
rq4 = safe_read_parquet('results/rq4_comparison.parquet')
if rq4 is None:
    print("Skipping RQ4 figure — results/rq4_comparison.parquet not found (run Notebook 06 first).")
else:
    rq4_acc_agg = rq4.groupby(['task_group', 'method'])['accuracy'].mean().reset_index()
    rq4_rho_agg = rq4.groupby(['task_group', 'method'])['rho_mean'].mean().reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=rq4_acc_agg, x='task_group', y='accuracy', hue='method', ax=axes[0])
    axes[0].set_title('RQ4: accuracy (mean over shift types)')
    axes[0].set_xlabel('Task group')

    sns.barplot(data=rq4_rho_agg, x='task_group', y='rho_mean', hue='method', ax=axes[1])
    axes[1].set_title('RQ4: correctness-of-reliance rho')
    axes[1].set_xlabel('Task group')
    axes[1].set_ylabel('rho(pi_behav, pi_true)')

    fig.tight_layout()
    fig.savefig(resolve_path('figures/rq4_sata_comparison.pdf'))
    plt.close(fig)
    print("Saved figures/rq4_sata_comparison.pdf")

Skipping RQ4 figure — results/rq4_comparison.parquet not found (run Notebook 06 first).


In [ ]:
from src.evaluation.uncertainty import confidence_from_logprobs, compute_rauc


def plot_retention(df, arm_name, ax):
    if df is None or df.empty:
        ax.set_title(f'{arm_name}: no data')
        return
    df = df.copy()
    df['confidence'] = confidence_from_logprobs(df['logprob_0'].to_numpy(), df['logprob_1'].to_numpy())

    best, worst, best_rauc, worst_rauc = None, None, np.inf, -np.inf
    for keys, group in df.groupby(['dataset', 'environment', 'model', 'method']):
        rauc, levels, errors = compute_rauc(
            group['prediction'].to_numpy(), group['label'].to_numpy(), group['confidence'].to_numpy()
        )
        if rauc < best_rauc:
            best_rauc, best = rauc, (keys, levels, errors)
        if rauc > worst_rauc:
            worst_rauc, worst = rauc, (keys, levels, errors)

    if best:
        ax.plot(best[1], best[2], label=f'best: {best[0][3]} (R-AUC={best_rauc:.3f})')
    if worst:
        ax.plot(worst[1], worst[2], label=f'worst: {worst[0][3]} (R-AUC={worst_rauc:.3f})')
    ax.set_title(arm_name)
    ax.set_xlabel('Retention')
    ax.set_ylabel('Error rate')
    ax.legend()


real_results_r8 = safe_read_parquet('results/real_arm_baselines.parquet')
synthetic_results_r8 = safe_read_parquet('results/synthetic_evaluation.parquet')

# Both notebooks now sweep k_primary + k_sensitivity, so these raw
# per-prediction files have rows at both k values for the same
# (dataset, environment, model, method) -- filter to k_primary so R-AUC is
# computed on one coherent set of predictions per group, not k=8 and k=16
# rows mixed together. zero_shot rows are always stored with k=0 (no demos,
# regardless of which k in the sweep was active) -- keep those too, or the
# filter would silently drop zero_shot out of the best/worst search.
if real_results_r8 is not None and 'k' in real_results_r8.columns:
    real_results_r8 = real_results_r8[
        (real_results_r8['k'] == config.k_primary) | (real_results_r8['method'] == 'zero_shot')
    ]
if synthetic_results_r8 is not None and 'k' in synthetic_results_r8.columns:
    synthetic_results_r8 = synthetic_results_r8[
        (synthetic_results_r8['k'] == config.k_primary) | (synthetic_results_r8['method'] == 'zero_shot')
    ]

if real_results_r8 is None and synthetic_results_r8 is None:
    print("Skipping retention curves figure — no results available yet (run Notebook 02 and/or 06 first).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_retention(real_results_r8, 'real arm', axes[0])
    plot_retention(synthetic_results_r8, 'synthetic arm', axes[1])
    fig.tight_layout()
    fig.savefig(resolve_path('figures/retention_curves.pdf'))
    plt.close(fig)
    print("Saved figures/retention_curves.pdf")

In [ ]:
# Full SATA vs query-agnostic vs best-protocol-alone, from rq2_grid (already
# covers all 3 as separate `method` values, at k_primary).
rq2_for_ablation = safe_read_parquet('results/rq2_grid.parquet')
if rq2_for_ablation is None:
    print("Skipping SATA ablation figure — results/rq2_grid.parquet not found (run Notebook 06 first).")
else:
    ablation_methods = ['sata_alone', 'sata_query_agnostic', 'best_protocol_sata']
    ablation_data = rq2_for_ablation[rq2_for_ablation['method'].isin(ablation_methods)]

    ablation_table = ablation_data.pivot_table(index=['model', 'method'], columns='shift_type', values='accuracy')
    ablation_table.to_csv(resolve_path('tables/sata_ablations.csv'))

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=ablation_data, x='shift_type', y='accuracy', hue='method', ax=ax)
    ax.set_title('SATA ablations: full vs query-agnostic vs best-protocol-alone')
    fig.tight_layout()
    fig.savefig(resolve_path('figures/sata_ablations.pdf'))
    plt.close(fig)
    print("Saved figures/sata_ablations.pdf + tables/sata_ablations.csv")

# k/pool-size sensitivity: real arm (Notebook 02) and synthetic arm
# (Notebook 06) both now sweep k_primary vs k_sensitivity for every
# condition -- surfaced here as its own table/figure rather than folded into
# the ablation figure above, since it's a different comparison axis (k, not
# method).
real_summary_for_k = safe_read_parquet('results/real_arm_baselines_summary.parquet')
rq2_k_sensitivity = safe_read_parquet('results/rq2_grid_k_sensitivity.parquet')

if real_summary_for_k is None and rq2_k_sensitivity is None:
    print("Skipping k-sensitivity figure — neither real_arm_baselines_summary.parquet nor "
          "rq2_grid_k_sensitivity.parquet found (run Notebook 02 and/or 06 first).")
else:
    k_rows = []
    if real_summary_for_k is not None and 'k' in real_summary_for_k.columns:
        for (dataset, model, method, k), grp in real_summary_for_k.groupby(['dataset', 'model', 'method', 'k']):
            k_rows.append({
                'arm': 'real', 'group': dataset, 'model': model, 'method': method, 'k': k,
                'accuracy': grp[grp['environment'] == 'ood']['accuracy_mean'].mean(),
            })
    if rq2_k_sensitivity is not None and 'k' in rq2_k_sensitivity.columns:
        for (shift_type, model, method, k), grp in rq2_k_sensitivity.groupby(['shift_type', 'model', 'method', 'k']):
            k_rows.append({
                'arm': 'synthetic', 'group': shift_type, 'model': model, 'method': method, 'k': k,
                'accuracy': grp['accuracy'].mean(),
            })

    if not k_rows or len({row['k'] for row in k_rows}) < 2:
        print("Skipping k-sensitivity figure — need both k_primary and k_sensitivity rows "
              "(re-run Notebook 02/06 with the k-sweep to populate them).")
    else:
        k_sensitivity_df = pd.DataFrame(k_rows)
        k_sensitivity_table = k_sensitivity_df.pivot_table(
            index=['arm', 'group', 'model', 'method'], columns='k', values='accuracy'
        )
        k_sensitivity_table.to_csv(resolve_path('tables/k_sensitivity.csv'))

        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        for ax, arm_name in zip(axes, ['real', 'synthetic']):
            arm_df = k_sensitivity_df[k_sensitivity_df['arm'] == arm_name]
            if arm_df.empty:
                ax.set_title(f'{arm_name}: no data')
                continue
            sns.barplot(data=arm_df, x='method', y='accuracy', hue='k', ax=ax)
            ax.set_title(f'k-sensitivity: {arm_name} arm')
            ax.tick_params(axis='x', rotation=45)
        fig.tight_layout()
        fig.savefig(resolve_path('figures/k_sensitivity.pdf'))
        plt.close(fig)
        print("Saved figures/k_sensitivity.pdf + tables/k_sensitivity.csv")

In [9]:
# Figure list item 7 ("training curves") -- not in the Output section's named
# PDF list but part of the numbered figure spec, so saved under its own name.
training_log = safe_read_parquet('results/sata_training_log.parquet')
if training_log is None:
    print("Skipping training curves figure — results/sata_training_log.parquet not found (run Notebook 05 first).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for model_name, grp in training_log.groupby('model'):
        axes[0].plot(grp['epoch'], grp['loss'], label=model_name)
        axes[1].plot(grp['epoch'], grp['val_proxy'], label=model_name)
    axes[0].set_title('SATA training loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('KL loss')
    axes[0].legend()
    axes[1].set_title('SATA validation proxy accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('XGBoost proxy accuracy')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(resolve_path('figures/sata_training_curves.pdf'))
    plt.close(fig)
    print("Saved figures/sata_training_curves.pdf")

Saved figures/sata_training_curves.pdf


## Statistical tests

- Paired Wilcoxon signed-rank test across tasks for protocol comparisons.
- Holm correction for multiple comparisons when comparing all protocols.
- All CIs from bootstrap (already computed in evaluation notebooks).

**Why paired and non-parametric, specifically.** Each synthetic task gives every condition a prediction on the *same* set of tasks, so a paired test (Wilcoxon signed-rank, on per-task accuracy) is more powerful than an unpaired test that ignores this structure — and non-parametric because there's no reason to assume per-task accuracy differences are normally distributed. Holm correction matters because this notebook runs many pairwise protocol comparisons at once (Notebook 06's 10 conditions); without correcting for multiple comparisons, some "significant" differences would be expected to appear by chance alone even if no real difference existed.

In [ ]:
from itertools import combinations

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

synthetic_results_r8b = safe_read_parquet('results/synthetic_evaluation.parquet')
if synthetic_results_r8b is None:
    print("Skipping Wilcoxon/Holm tests — results/synthetic_evaluation.parquet not found (run Notebook 06 first).")
    wilcoxon_df = pd.DataFrame(columns=['method_a', 'method_b', 'statistic', 'pval', 'pval_holm', 'significant'])
else:
    # Notebook 06 now sweeps k_primary + k_sensitivity, so this file has rows
    # at both k values per (dataset, method) -- filter to k_primary (keeping
    # zero_shot, always stored at k=0) so per_task below pairs each method's
    # k_primary accuracy, not a blend of both k values.
    if 'k' in synthetic_results_r8b.columns:
        synthetic_results_r8b = synthetic_results_r8b[
            (synthetic_results_r8b['k'] == config.k_primary) | (synthetic_results_r8b['method'] == 'zero_shot')
        ]

    # Paired "across tasks" comparison, per the spec -- only the synthetic arm
    # has discrete per-task units to pair on (the real arm's unit is rows
    # within a dataset, not a comparable "task"). Uses the 'id' shift type as
    # the common basis across methods.
    id_results = synthetic_results_r8b[synthetic_results_r8b['environment'] == 'id'].copy()
    id_results['correct'] = (id_results['prediction'] == id_results['label']).astype(int)
    per_task = id_results.groupby(['dataset', 'method'])['correct'].mean().reset_index()
    wide = per_task.pivot(index='dataset', columns='method', values='correct').dropna()

    pairs, stats, pvals = [], [], []
    for a, b in combinations(wide.columns, 2):
        diff = wide[a] - wide[b]
        if (diff == 0).all():
            continue
        stat, p = wilcoxon(wide[a], wide[b])
        pairs.append((a, b))
        stats.append(stat)
        pvals.append(p)

    if not pvals:
        print("No valid protocol pairs to test (identical accuracy across all tasks, or too few tasks).")
        wilcoxon_df = pd.DataFrame(columns=['method_a', 'method_b', 'statistic', 'pval', 'pval_holm', 'significant'])
    else:
        reject, pvals_holm, _, _ = multipletests(pvals, method='holm')
        wilcoxon_df = pd.DataFrame({
            'method_a': [p[0] for p in pairs],
            'method_b': [p[1] for p in pairs],
            'statistic': stats,
            'pval': pvals,
            'pval_holm': pvals_holm,
            'significant': reject,
        })

wilcoxon_df.to_csv(resolve_path('tables/wilcoxon_holm.csv'), index=False)
wilcoxon_df

## Output

- `figures/*.pdf`
- `tables/` — LaTeX table source files for thesis